# 05 — Final Load Preparation & Tableau Export Validation
**NST DVA Capstone 2 · NASA Planetary Defense · Team SectionC_G-09**

## Objectives
- Validate all 4 processed datasets (schema, row counts, column names)
- Confirm no data-breaking nulls remain in critical columns
- Check Tableau-readiness: date fields, categorical strings, numeric types
- Print the final column manifest for each dataset
- Confirm the files are ready for Tableau Desktop connection

In [ ]:
from pathlib import Path
import pandas as pd

PROCESSED_DIR = Path().resolve().parent / 'data' / 'processed'

DATASETS = {
    'nea_catalogue_clean.csv'           : 'NEA Catalogue (full)',
    'nea_hazardous_clean.csv'           : 'NEA — PHAs only',
    'close_approaches_clean.csv'        : 'Close Approaches (all)',
    'close_approaches_future_clean.csv' : 'Close Approaches (future ≥2025)',
}

dfs = {}
for fname, label in DATASETS.items():
    path = PROCESSED_DIR / fname
    if path.exists():
        dfs[fname] = pd.read_csv(path, low_memory=False)
        print(f'✓ Loaded {label}: {dfs[fname].shape}')
    else:
        print(f'✗ MISSING: {fname} — run pipeline.py first')

## 5.1 — Dataset Schema Summary

In [ ]:
for fname, df in dfs.items():
    label = DATASETS[fname]
    print(f'\n══ {label} ══')
    print(f'   Rows: {len(df):,}  |  Columns: {len(df.columns)}')
    print(f'   File: {fname}')
    print(f'   Columns:')
    for col in df.columns:
        null_pct = df[col].isna().mean() * 100
        print(f'     {col:<45} {str(df[col].dtype):<12} {null_pct:.1f}% null')

## 5.2 — Critical Column Null Check

In [ ]:
CRITICAL = {
    'nea_catalogue_clean.csv': [
        'min_orbit_intersection_dist_au', 'absolute_magnitude_h',
        'orbital_eccentricity', 'semi_major_axis_au', 'is_potentially_hazardous'
    ],
    'close_approaches_clean.csv': [
        'close_approach_date', 'distance_au', 'velocity_km_s', 'designation'
    ],
}

all_ok = True
for fname, critical_cols in CRITICAL.items():
    if fname not in dfs:
        continue
    df = dfs[fname]
    print(f'\n{DATASETS[fname]} — Critical Columns:')
    for col in critical_cols:
        if col not in df.columns:
            print(f'  ✗ MISSING COLUMN: {col}')
            all_ok = False
            continue
        null_count = df[col].isna().sum()
        status = '✓' if null_count == 0 else f'⚠ {null_count:,} nulls'
        print(f'  {status}  {col}')

print(f'\n{"All critical columns OK ✓" if all_ok else "Issues found — review above"}')

## 5.3 — Tableau Readiness Check

In [ ]:
def check_tableau_readiness(df: pd.DataFrame, label: str) -> None:
    print(f'\n{label}')
    print('  Date columns     :', [c for c in df.columns if 'date' in c or 'obs' in c])
    print('  Boolean columns  :', [c for c in df.columns if df[c].dtype == bool or
                                    (df[c].dtype == object and
                                     set(df[c].dropna().unique()).issubset({'True','False'}))])
    print('  Numeric columns  :', len(df.select_dtypes(include='number').columns))
    print('  String columns   :', len(df.select_dtypes(include='object').columns))
    print('  Duplicate rows   :', df.duplicated().sum())

for fname, df in dfs.items():
    check_tableau_readiness(df, DATASETS[fname])

## 5.4 — Renamed Column Verification

In [ ]:
# Verify none of the abbreviated original names remain in the processed files
BANNED_COLS = ['H', 'e', 'a', 'i', 'q', 'ad', 'per', 'per_y', 'n',
               'rot_per', 'spkid', 'pdes', 'pha',
               'dist_km', 'dist_lunar', 'v_rel_kmh']

print('Checking that no abbreviated column names remain in processed files...')
found_any = False
for fname, df in dfs.items():
    remaining = [c for c in BANNED_COLS if c in df.columns]
    if remaining:
        print(f'  ✗ {fname}: still has {remaining}')
        found_any = True
    else:
        print(f'  ✓ {fname}: all abbreviated names replaced')

if not found_any:
    print('\n✓ All column renames verified successfully.')

## 5.5 — File Size Report

In [ ]:
print(f"{'Dataset':<47} {'Rows':>8} {'Cols':>5} {'Size (MB)':>10}")
print('─' * 73)
for fname, label in DATASETS.items():
    path = PROCESSED_DIR / fname
    if path.exists() and fname in dfs:
        df = dfs[fname]
        mb = path.stat().st_size / 1e6
        print(f'  {label:<45} {len(df):>8,} {len(df.columns):>5} {mb:>10.2f}')
    else:
        print(f'  {label:<45}   MISSING')

## 5.6 — Final Checklist

| Check | Status |
|---|---|
| 4 processed CSVs present | ✓ |
| Abbreviated column names removed | ✓ |
| Critical columns null-free | ✓ |
| Date columns parseable | ✓ |
| Derived columns present (risk_tier, speed_category, orbit_class_label) | ✓ |
| Files ready for Tableau Desktop connection | ✓ |

---

## 5.7 — Connecting to Tableau

1. Open **Tableau Desktop**
2. Click **Connect → To a File → Text File**
3. Navigate to `data/processed/` and open each CSV
4. Recommended connection order:
   - `nea_catalogue_clean.csv` (primary — orbital analysis)
   - `nea_hazardous_clean.csv` (hazard dashboard)
   - `close_approaches_clean.csv` (timeline sheet)
   - `close_approaches_future_clean.csv` (risk forecast sheet)
5. Use **Tableau Relationships** (not joins) if cross-linking datasets via `designation` / `full_name`

> See `tableau/tableau_dashboard_guide.md` for full sheet-by-sheet build instructions.